# Generating Simulated Data

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime , timedelta

num_customers = 75000
num_trasactions = 300000
start_date = "2023-01-01"
end_date = "2024-09-30"
product_categories = [
                        'Groceries',
                        'Dining Out & Food Delivery',
                        'Transportation',
                        'Utilities',
                        'Mobile & Internet Bills',
                        'Shopping & E-commerce',
                        'Fuel',
                        'Rent',
                        'Health & Pharmacy',
                        'Entertainment',
                        'Subscriptions',
                        'Personal Care',
                        'Travel',
                        'Education & Fees',
                        'Home Supplies & Services',
                        'Gifts & Donations'
                    ]

customer_ids = range(1 , num_customers + 1 )
transaction_dates = pd.to_datetime(np.random.randint(
    pd.to_datetime(start_date).value // 10**9,
    pd.to_datetime(end_date).value // 10**9,
    num_trasactions
), unit='s')

data = {
    'CustomerID' : np.random.choice(customer_ids , num_trasactions),
    'TransactionDate' : transaction_dates,
    'ProductCategory' : np.random.choice(product_categories , num_trasactions),
    'TransactionAmount' : np.round(np.random.gamma(2,50,num_trasactions),2)
}

df = pd.DataFrame(data)

champions = np.random.choice(customer_ids , size = int(num_customers*0.1), replace=False)
df.loc[df['CustomerID'].isin(champions),'TransactionAmount'] *= 2.5

at_risk_customers = np.random.choice(customer_ids, size = int(num_customers*0.15), replace = False)
at_risk_indices = df[df['CustomerID'].isin(at_risk_customers)].index
df.loc[at_risk_indices, 'TransactionDate'] -= timedelta(days=180)
df.to_csv('transaction.csv',index = False)
print('--- the data is generated sucessfully ---')

--- the data is generated sucessfully ---


In [2]:
df

,CustomerID,TransactionDate,ProductCategory,TransactionAmount
0,19769,2023-12-21 22:53:15,Home Supplies & Services,168.08
1,51375,2024-03-11 08:36:31,Entertainment,23.51
2,69931,2023-12-24 23:28:20,Fuel,37.97
3,54222,2023-11-14 19:41:06,Utilities,26.54
4,28814,2024-09-02 07:54:39,Entertainment,61.00
...,...,...,...,...
299995,47135,2023-03-19 15:17:04,Gifts & Donations,143.46
299996,53345,2023-09-23 10:10:27,Fuel,28.02
299997,69229,2024-03-14 09:44:07,Home Supplies & Services,19.43
299998,53965,2022-07-05 21:01:54,Education & Fees,70.78


In [3]:
import pandas as pd
from datetime import datetime

df = pd.read_csv('transaction.csv')

# Calculating RFM :- Recency , Frequency , Monetary

In [4]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

snapshot_date = df['TransactionDate'].max() + timedelta(days=1)

rfm_data = df.groupby('CustomerID').agg({
    'TransactionDate' : lambda date : (snapshot_date - date.max()).days,
    'CustomerID' : 'count',
    'TransactionAmount' : 'sum'
})

rfm_data.rename(columns={
    'TransactionDate' : 'Recency',
    'CustomerID' : 'Frequency',
    'TransactionAmount' : 'Monetary'
},inplace=True)

print(rfm_data.head())

            Recency  Frequency  Monetary
CustomerID                              
1               190          5    347.15
2               412          2    349.51
3                 2          8    739.74
4               332          4    438.15
5                49          4    422.86


# Creating Predictive CLV & Smart Segmentation  

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

x = rfm_data[['Recency','Frequency']]
y = rfm_data['Monetary']

scaler = StandardScaler()
x_scaled  = scaler.fit_transform(x)

model = LinearRegression()
model.fit(x_scaled , y)

rfm_data['PredictiveCLV'] = model.predict(x_scaled)

rfm_data['PredictiveCLV'] = rfm_data['PredictiveCLV'].apply(lambda x : max(x,0))

In [6]:
from sklearn.cluster import KMeans

features_for_clustering = rfm_data[['Recency','Frequency','Monetary','PredictiveCLV']]
scaled_features = scaler.fit_transform(features_for_clustering)

kmeans = KMeans(n_clusters=4 , random_state=42 , n_init=10)
rfm_data['Segment'] = kmeans.fit_predict(scaled_features)

segment_analysis = rfm_data.groupby('Segment').agg({
    'Recency' : 'mean',
    'Frequency' : 'mean',
    'Monetary' : 'mean',
    'PredictiveCLV' : 'mean'
}).sort_values(by='PredictiveCLV',ascending=False)

segment_map = {
    segment_analysis.index[0]: '🏆 Champions',
    segment_analysis.index[1]: '📈 Loyalists',
    segment_analysis.index[2]: '❗ Needs Attention',
    segment_analysis.index[3]: '📉 At Risk',
}

rfm_data['SegmentName'] = rfm_data['Segment'].map(segment_map)
print(segment_analysis)

            Recency  Frequency    Monetary  PredictiveCLV
Segment                                                  
2        104.566345   7.110953  980.917639     818.230109
1        128.058141   4.708924  494.313150     542.248941
3        127.419108   2.449917  264.837122     282.955804
0        420.011944   2.247188  242.960418     256.395527


In [7]:
rfm_data.to_csv('RFMDATA.csv')

In [8]:
full_data = df.merge(rfm_data,on='CustomerID')

category_prefs = full_data.groupby(['SegmentName','ProductCategory']).size().reset_index(name='Count')
top_categories = category_prefs.loc[category_prefs.groupby('SegmentName')['Count'].idxmax()]

print(top_categories)

          SegmentName           ProductCategory  Count
2   ❗ Needs Attention             Entertainment   3042
19        🏆 Champions                      Fuel   5473
39        📈 Loyalists  Home Supplies & Services   8842
57          📉 At Risk             Personal Care   1890


In [9]:
top_categories

,SegmentName,ProductCategory,Count
2,❗ Needs Attention,Entertainment,3042
19,🏆 Champions,Fuel,5473
39,📈 Loyalists,Home Supplies & Services,8842
57,📉 At Risk,Personal Care,1890


In [14]:
def recommend_offer(row):
  segment = row['SegmentName']
  try :
    top_cat = top_categories[top_categories['SegmentName']==segment]['ProductCategory'].iloc[0]
  except IndexError:
    top_cat = 'General'

  if segment == '🏆 Champions':
    return f'Exclusive Preview : 15% off new arrivals in {top_cat}!'
  elif segment == '📈 Loyalists':
    return f'Get upto 2X points on {top_cat} purchases this week! '
  elif segment == '❗ Needs Attention':
    return f'We miss you here is ₹500 credit for your next purchase.'
  elif segment == '📉 At Risk':
    return f'Come back and get upto 20% OFF your entire next purchase.'
  else :
    return f'Enjoy Free Shopping on us!'

rfm_data['RecommendedOffer'] = rfm_data.apply(recommend_offer,axis=1)
output_df = rfm_data.reset_index()
output_df.to_csv('customer_segments_with_offers.csv',index=False)
print('---- Final dataset with recommendation is ready for visualization ----')

---- Final dataset with recommendation is ready for visualization ----


In [15]:
output_df

,CustomerID,Recency,Frequency,Monetary,PredictiveCLV,Segment,SegmentName,RecommendedOffer
0,1,190,5,347.15,574.963595,1,📈 Loyalists,Get upto 2X points on Home Supplies & Services...
1,2,412,2,349.51,228.112078,0,📉 At Risk,Come back and get upto 20% OFF your entire nex...
2,3,2,8,739.74,921.432797,2,🏆 Champions,Exclusive Preview : 15% off new arrivals in Fuel!
3,4,332,4,438.15,458.581793,1,📈 Loyalists,Get upto 2X points on Home Supplies & Services...
4,5,49,4,422.86,461.764000,1,📈 Loyalists,Get upto 2X points on Home Supplies & Services...
...,...,...,...,...,...,...,...,...
73600,74996,503,3,127.45,341.873900,0,📉 At Risk,Come back and get upto 20% OFF your entire nex...
73601,74997,306,3,135.94,344.089076,0,📉 At Risk,Come back and get upto 20% OFF your entire nex...
73602,74998,137,2,199.07,231.204329,3,❗ Needs Attention,We miss you here is ₹500 credit for your next ...
73603,74999,260,5,575.65,574.176477,1,📈 Loyalists,Get upto 2X points on Home Supplies & Services...
